# Notebook 13 – Model Performance Experiment

**File:** `13_Model_Performance_Experiment.ipynb`

## 1. Introduction to Model Performance Experiment

A model performance experiment helps us understand how efficiently a pre-trained model works when making predictions.

A model may produce accurate results but require more time, memory, or computing power.

In this experiment, we will compare at least two pre-trained models performing the same task.

We will measure:

- Inference time
- Model size
- Memory usage, where practical
- CPU performance
- GPU performance, if available
- Output quality

**Goal:** Understand that model quality is not the only consideration when selecting a pre-trained model.

In [1]:
import time
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.14.0+cpu
GPU available: False


# 2. Inference Time

**Inference time** is the time a trained model takes to process input data and produce an output.

For example, when an image classification model receives a picture of a cat, inference time measures how long the model takes to predict the image's class.

**Why do we measure it?**

- To understand how quickly a model produces predictions.
- To compare the speed of two pre-trained models.
- To select a suitable model for applications that require fast responses.

**Important:** Lower inference time means faster prediction, provided we compare models using the same hardware, input, and measurement method.

Inference time can vary depending on the model, input size, CPU/GPU, and system load.

In [2]:
import time

start_time = time.perf_counter()

# Simulate a small amount of processing
total = sum(range(1_000_000))

end_time = time.perf_counter()

inference_time = end_time - start_time

print("Result:", total)
print("Execution time:", round(inference_time, 4), "seconds")

Result: 499999500000
Execution time: 0.05 seconds


# 3. Model Size

**Model size** is the amount of storage space required to save a trained model and its parameters.

A pre-trained model contains learned parameters (weights and biases) that are used to generate predictions.

For example, one model might occupy 50 MB, while another might occupy 500 MB.

**Why do we measure it?**

- To understand how much disk space a model requires.
- To compare lightweight and large models.
- To determine whether a model is suitable for a laptop or a device with limited storage.

**Important:** A smaller model is generally easier to store and distribute, but smaller size does not automatically mean faster inference or better output quality.

Model size also depends on the weight format and precision, such as FP32 or FP16.

In [3]:
import os

file_path = "sample_model.pth"

# Create a small sample file for demonstration
with open(file_path, "wb") as file:
    file.write(b"0" * 1024 * 1024)

file_size_bytes = os.path.getsize(file_path)
file_size_mb = file_size_bytes / (1024 * 1024)

print("Model file size:", round(file_size_mb, 2), "MB")

Model file size: 1.0 MB


# 4. Memory Usage

**Memory usage** is the amount of RAM required while a model is loaded and performing predictions.

A pre-trained model needs memory to store its parameters, process input data, and produce outputs.

For example, one model may use 300 MB of RAM, while another may use 1 GB.

**Why do we measure it?**

- To understand how much RAM a model requires.
- To compare the resource requirements of different models.
- To determine whether a model can run comfortably on a laptop.

**Important:** Model file size and memory usage are different. A model's RAM usage may be higher than its file size because additional memory is needed during processing.

Memory usage depends on the model, input size, batch size, and execution environment.

In [4]:
import os
import psutil

process = psutil.Process(os.getpid())

memory_mb = process.memory_info().rss / (1024 * 1024)

print("Current process memory:", round(memory_mb, 2), "MB")

Current process memory: 77.3 MB


# 5. CPU Performance

**CPU performance** describes how efficiently a pre-trained model performs predictions using the computer's central processing unit (CPU).

A CPU can run many pre-trained models without requiring a dedicated GPU.

In our experiment, we will run models on the CPU because our PyTorch installation is `2.14.0+cpu` and GPU availability is `False`.

**Why do we measure it?**

- To understand how quickly a model runs on a CPU.
- To compare the inference time of two models on the same laptop.
- To identify models suitable for computers without a GPU.

**Important:** A model with fewer parameters may run faster, but actual speed also depends on model architecture, input size, CPU capabilities, and other factors.

In [5]:
import os
import platform
import torch

print("Processor:", platform.processor())
print("Logical CPU cores:", os.cpu_count())
print("PyTorch CPU threads:", torch.get_num_threads())

Processor: Intel64 Family 6 Model 186 Stepping 3, GenuineIntel
Logical CPU cores: 8
PyTorch CPU threads: 6


# 6. GPU Performance

**GPU performance** describes how efficiently a pre-trained model performs predictions using a graphics processing unit (GPU).

GPUs can process many operations in parallel, making them useful for deep learning tasks.

**Why do we measure it?**

- To compare model inference speed on a CPU and GPU.
- To understand whether GPU acceleration improves performance.
- To identify models that benefit from GPU resources.

**Important:** A GPU does not guarantee faster inference for every model. Performance depends on the model, input size, GPU hardware, and the overhead of transferring data between the CPU and GPU.

In this experiment, GPU benchmarking is optional. Your current PyTorch installation is CPU-only, so we will perform the actual comparison using your CPU.

In [6]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU available:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("GPU unavailable. Using CPU.")

print("Selected device:", device)

GPU unavailable. Using CPU.
Selected device: cpu


# 7. Output Quality

**Output quality** describes how correctly and usefully a pre-trained model performs its task.

For example, two image classification models may process the same image but produce different predictions.

- Model A predicts `cat` with a score of 0.95.
- Model B predicts `cat` with a score of 0.78.

We should also check whether the predicted class is correct.

**Why do we measure it?**

- To compare the correctness of model predictions.
- To understand the differences between two models.
- To avoid selecting a model based only on speed or size.

**Important:** A higher confidence score does not always mean a prediction is more accurate. Output quality should be evaluated against known correct answers when possible.

For a fair experiment, both models must perform the same task using the same test inputs.

In [7]:
model_a = {
    "prediction": "cat",
    "confidence": 0.95
}

model_b = {
    "prediction": "cat",
    "confidence": 0.78
}

correct_label = "cat"

for name, result in [
    ("Model A", model_a),
    ("Model B", model_b)
]:
    is_correct = result["prediction"] == correct_label

    print(name)
    print("Prediction:", result["prediction"])
    print("Confidence:", result["confidence"])
    print("Correct:", is_correct)
    print()

Model A
Prediction: cat
Confidence: 0.95
Correct: True

Model B
Prediction: cat
Confidence: 0.78
Correct: True



# 8. Comparing Two Models for the Same Task

To understand practical model performance, we must compare at least two pre-trained models that perform the same task.

For this experiment, we will use **image classification**.

We will compare:

- **Model A:** MobileNetV2 — a lightweight image classification model.
- **Model B:** ResNet50 — a deeper image classification model.

Both models can classify images into categories, making them suitable for comparison.

We will measure:

1. **Parameters:** The number of learned values in the model.
2. **Inference time:** How long the model takes to predict an image.
3. **Model size:** The storage space required for the model weights.
4. **Resource usage:** The RAM used while running the model.
5. **Output quality:** Whether the model correctly classifies test images.

Both models will run on the CPU because GPU acceleration is unavailable in our current environment.

**Important:** We must use the same test images and comparable preprocessing for both models to make the comparison fair.

In [8]:
import torch
from torchvision import models

model_a = models.mobilenet_v2(weights="DEFAULT")
model_b = models.resnet50(weights="DEFAULT")

model_a.eval()
model_b.eval()

print("Model A: MobileNetV2 loaded")
print("Model B: ResNet50 loaded")
print("Device: CPU")

Model A: MobileNetV2 loaded
Model B: ResNet50 loaded
Device: CPU


# 9. Measuring Model Parameters and Size

A model's **parameter count** tells us how many learned values it contains. Model size tells us how much storage space its saved weights require.

In our experiment, we will measure both MobileNetV2 and ResNet50.

- **Parameter count:** Number of learned weights and biases.
- **Model size:** Storage required for the model's parameters when saved to a file.
- **Trainable parameters:** Parameters that can be updated during training. For pretrained inference, we generally do not update them.

A model with more parameters often requires more storage and computation, but parameter count alone does not determine inference speed or output quality.

In [9]:
import torch

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def measure_model_size(model):
    total_bytes = sum(
        p.numel() * p.element_size()
        for p in model.parameters()
    )
    return total_bytes / (1024 * 1024)

for name, model in [
    ("MobileNetV2", model_a),
    ("ResNet50", model_b)
]:
    parameters = count_parameters(model)
    size_mb = measure_model_size(model)

    print(name)
    print("Parameters:", f"{parameters:,}")
    print("Parameter size:", round(size_mb, 2), "MB")
    print()

MobileNetV2
Parameters: 3,504,872
Parameter size: 13.37 MB

ResNet50
Parameters: 25,557,032
Parameter size: 97.49 MB



In [10]:
import torch

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def measure_model_size(model):
    total_bytes = sum(
        p.numel() * p.element_size()
        for p in model.parameters()
    )
    return total_bytes / (1024 * 1024)

for name, model in [
    ("MobileNetV2", model_a),
    ("ResNet50", model_b)
]:
    parameters = count_parameters(model)
    size_mb = measure_model_size(model)

    print(name)
    print("Parameters:", f"{parameters:,}")
    print("Parameter size:", round(size_mb, 2), "MB")
    print()

MobileNetV2
Parameters: 3,504,872
Parameter size: 13.37 MB

ResNet50
Parameters: 25,557,032
Parameter size: 97.49 MB



In [11]:
%pip install psutil


Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
import gc
import psutil
import torch

process = psutil.Process(os.getpid())

def get_memory_mb():
    return process.memory_info().rss / (1024 * 1024)

sample_image = torch.randn(1, 3, 224, 224)

for name, model in [
    ("MobileNetV2", model_a),
    ("ResNet50", model_b)
]:
    gc.collect()
    model.eval()

    memory_before = get_memory_mb()

    with torch.inference_mode():
        output = model(sample_image)

    memory_after = get_memory_mb()

    print(name)
    print("Memory before inference:",
          round(memory_before, 2), "MB")
    print("Memory after inference:",
          round(memory_after, 2), "MB")
    print("Observed difference:",
          round(memory_after - memory_before, 2), "MB")
    print()

MobileNetV2
Memory before inference: 299.76 MB
Memory after inference: 321.24 MB
Observed difference: 21.48 MB

ResNet50
Memory before inference: 321.24 MB
Memory after inference: 414.43 MB
Observed difference: 93.19 MB



In [13]:
import time
import torch

sample_image = torch.randn(1, 3, 224, 224)

models_to_compare = [
    ("MobileNetV2", model_a),
    ("ResNet50", model_b)
]

results = {}

for name, model in models_to_compare:
    model = model.to("cpu")
    model.eval()

    with torch.inference_mode():
        for _ in range(3):
            model(sample_image)

        start = time.perf_counter()

        for _ in range(20):
            model(sample_image)

        end = time.perf_counter()

    avg_ms = (end - start) / 20 * 1000
    results[name] = avg_ms

for name, avg_ms in results.items():
    print(name, "CPU inference:",
          round(avg_ms, 2), "ms")

if results["MobileNetV2"] < results["ResNet50"]:
    print("MobileNetV2 was faster in this test.")
elif results["ResNet50"] < results["MobileNetV2"]:
    print("ResNet50 was faster in this test.")
else:
    print("Both models had equal measured times.")

MobileNetV2 CPU inference: 74.2 ms
ResNet50 CPU inference: 252.79 ms
MobileNetV2 was faster in this test.


In [14]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))

    for name, model in [
        ("MobileNetV2", model_a),
        ("ResNet50", model_b)
    ]:
        model = model.to(device).eval()
        sample_image = torch.randn(
            1, 3, 224, 224, device=device
        )

        with torch.inference_mode():
            for _ in range(3):
                model(sample_image)

            torch.cuda.synchronize()
            start = time.perf_counter()

            for _ in range(20):
                model(sample_image)

            torch.cuda.synchronize()
            end = time.perf_counter()

        print(
            name,
            "GPU inference:",
            round((end - start) / 20 * 1000, 2),
            "ms"
        )
else:
    print("GPU benchmarking skipped.")
    print("Reason: CUDA GPU is unavailable.")

GPU benchmarking skipped.
Reason: CUDA GPU is unavailable.


In [15]:
import torch

sample_image = torch.randn(1, 3, 224, 224)

models_to_compare = [
    ("MobileNetV2", model_a),
    ("ResNet50", model_b)
]

for name, model in models_to_compare:
    model.eval()

    with torch.inference_mode():
        logits = model(sample_image)
        probabilities = torch.softmax(logits, dim=1)

        confidence, predicted_class = probabilities.max(dim=1)

    print(name)
    print("Predicted class index:", predicted_class.item())
    print("Confidence score:", round(confidence.item(), 4))
    print()

MobileNetV2
Predicted class index: 92
Confidence score: 0.0122

ResNet50
Predicted class index: 21
Confidence score: 0.0957

